# Phase 05 — Model Training and Evaluation

This notebook runs the planned experiment matrix for ClearFlow:

- **10 feature sets** from Phase 04: 3 feature-selection methods × top 10/20/40 features, plus the 70-feature baseline.
- **4 classifiers:** Decision Tree, Random Forest, XGBoost, and Logistic Regression.
- **5 fixed, distinct random seeds:** 425828, 142707, 692819, 2184, 136548.
- **200 configurations** in total, unless a valid checkpoint is resumed.

The same class-capped training sample is reused for every configuration. The test split is used only for evaluation. The notebook records fit, inference, evaluation, total configuration and model-save times, along with sampled peak process RAM. RAM values are process-level measurements, not model-only memory.

Run the cells in order. The notebook expects the Phase 03 and Phase 04 outputs in the existing ClearFlow project structure.

## 1. Imports and project paths

The notebook searches the current directory and its parents for the expected Phase 03 manifest. If it cannot find the project root, set `PROJECT_ROOT` manually in the next cell.

In [2]:
from pathlib import Path
import json
import os
import re
import time
import threading
import warnings

import joblib
import numpy as np
import pandas as pd
import psutil

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False

SEED_MASTER_RANDOM_STATE = 20261022
DEFAULT_SEEDS = (425828, 142707, 692819, 2184, 136548)
SAMPLE_RANDOM_STATE = 42
MAX_ROWS_PER_CLASS = 10_000
CHUNK_SIZE = 100_000

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p.resolve() for p in candidate_roots
     if (p / "exports/phase03_split/split_manifest.json").exists()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate ClearFlow project root. Set PROJECT_ROOT manually "
        "to the folder containing exports/phase03_split/split_manifest.json."
    )

PHASE03_DIR = PROJECT_ROOT / "exports" / "phase03_split"
PHASE04_DIR = PROJECT_ROOT / "exports" / "phase04_feature_selection"
PHASE05_DIR = PROJECT_ROOT / "exports" / "phase05_model_results"
VISUALIZATION_DIR = PROJECT_ROOT / "exports" / "phase05_visualization"
MODEL_DIR = PROJECT_ROOT / "results" / "models" / "phase05"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train.csv.gz"
TEST_PATH = PROCESSED_DIR / "test.csv.gz"
SPLIT_MANIFEST_PATH = PHASE03_DIR / "split_manifest.json"
FEATURE_SETS_PATH = PHASE04_DIR / "feature_sets.json"

for directory in (PHASE05_DIR, VISUALIZATION_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("XGBoost available:", XGBOOST_AVAILABLE)
print("Training split exists:", TRAIN_PATH.exists())
print("Test split exists:", TEST_PATH.exists())
print("Split manifest exists:", SPLIT_MANIFEST_PATH.exists())
print("Feature sets file exists:", FEATURE_SETS_PATH.exists())

Project root: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow
XGBoost available: True
Training split exists: True
Test split exists: True
Split manifest exists: True
Feature sets file exists: True


## 2. Validate the Phase 03 split and Phase 04 feature sets

This cell validates the expected inputs before any model training starts. The target column and class order are taken from the split manifest when available.

In [3]:
if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("Expected train.csv.gz and test.csv.gz under data/processed/.")
if not SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing split manifest: {SPLIT_MANIFEST_PATH}")
if not FEATURE_SETS_PATH.exists():
    raise FileNotFoundError(f"Missing Phase 04 feature sets: {FEATURE_SETS_PATH}")

with open(SPLIT_MANIFEST_PATH, "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

with open(FEATURE_SETS_PATH, "r", encoding="utf-8") as f:
    feature_sets_payload = json.load(f)

TARGET_COLUMN = split_manifest.get("target_column", "Label")
MODEL_FEATURES = split_manifest.get("model_features")
CLASS_ORDER = split_manifest.get("class_order")

if not MODEL_FEATURES:
    raise ValueError("The split manifest must contain a non-empty 'model_features' list.")
if not CLASS_ORDER:
    raise ValueError("The split manifest must contain a non-empty 'class_order' list.")
if len(MODEL_FEATURES) != len(set(MODEL_FEATURES)):
    raise ValueError("Duplicate feature names found in split manifest.")

raw_feature_sets = feature_sets_payload.get("feature_sets", feature_sets_payload)
if isinstance(raw_feature_sets, dict):
    raw_feature_sets = list(raw_feature_sets.values())
if not isinstance(raw_feature_sets, list) or not raw_feature_sets:
    raise ValueError("Could not find a non-empty list of feature sets in feature_sets.json.")

def _first_present(mapping, names, default=None):
    for name in names:
        if name in mapping and mapping[name] is not None:
            return mapping[name]
    return default

FEATURE_SETS = []
for i, item in enumerate(raw_feature_sets, start=1):
    if not isinstance(item, dict):
        raise ValueError(f"Feature set entry {i} is not an object.")
    features = _first_present(item, ("features", "selected_features", "feature_names"))
    if not features:
        raise ValueError(f"Feature set entry {i} has no feature list.")
    features = list(features)
    unknown = sorted(set(features) - set(MODEL_FEATURES))
    if unknown:
        raise ValueError(f"Feature set entry {i} contains features outside Phase 03 model_features: {unknown[:5]}")
    feature_set_id = str(_first_present(item, ("feature_set_id", "id"), f"feature_set_{i:02d}"))
    feature_set_name = str(_first_present(item, ("feature_set_name", "name"), feature_set_id))
    selection_method = str(_first_present(item, ("selection_method", "method"), "unspecified"))
    FEATURE_SETS.append({
        "feature_set_id": feature_set_id,
        "feature_set_name": feature_set_name,
        "selection_method": selection_method,
        "features": features,
        "feature_count": len(features),
    })

if len(FEATURE_SETS) != 10:
    raise ValueError(f"Expected 10 Phase 04 feature sets, found {len(FEATURE_SETS)}.")

if len(CLASS_ORDER) != len(set(CLASS_ORDER)):
    raise ValueError("Duplicate labels found in class_order.")

# Read only headers here; this catches naming/schema mismatches without loading the full splits.
train_columns = pd.read_csv(TRAIN_PATH, nrows=0).columns.tolist()
test_columns = pd.read_csv(TEST_PATH, nrows=0).columns.tolist()
required_columns = set(MODEL_FEATURES) | {TARGET_COLUMN}
for split_name, columns in (("train", train_columns), ("test", test_columns)):
    missing = sorted(required_columns - set(columns))
    if missing:
        raise ValueError(f"{split_name} split is missing required columns: {missing[:10]}")

feature_set_table = pd.DataFrame([
    {
        "feature_set_id": fs["feature_set_id"],
        "feature_set_name": fs["feature_set_name"],
        "selection_method": fs["selection_method"],
        "feature_count": fs["feature_count"],
    }
    for fs in FEATURE_SETS
]).sort_values(["feature_count", "feature_set_id"])

print("Target column:", TARGET_COLUMN)
print("Model feature count:", len(MODEL_FEATURES))
print("Class count:", len(CLASS_ORDER))
print("Feature-set count:", len(FEATURE_SETS))
display(feature_set_table)

Target column: Label
Model feature count: 70
Class count: 12
Feature-set count: 10


,feature_set_id,feature_set_name,selection_method,feature_count
3,anova_top10,Top 10 — ANOVA F-test,anova_f_test,10
0,rf_top10,Top 10 — Random Forest Importance,rf_importance,10
6,rfe_top10,Top 10 — RFE (Decision Tree),rfe_decision_tree,10
4,anova_top20,Top 20 — ANOVA F-test,anova_f_test,20
1,rf_top20,Top 20 — Random Forest Importance,rf_importance,20
7,rfe_top20,Top 20 — RFE (Decision Tree),rfe_decision_tree,20
5,anova_top40,Top 40 — ANOVA F-test,anova_f_test,40
2,rf_top40,Top 40 — Random Forest Importance,rf_importance,40
8,rfe_top40,Top 40 — RFE (Decision Tree),rfe_decision_tree,40
9,all_features,All 70 Features (Shared Baseline),shared_all_features,70


## 3. Sampling, memory monitoring, and evaluation helpers

A single fixed class-capped sample is drawn from the training split and reused across all 200 configurations. Sampling is performed by chunks so the entire training CSV does not need to be loaded into memory. The test split is loaded once for consistent evaluation.

In [4]:
class PeakRSSMonitor:
    """Sample this Python process's resident memory; this is not model-only RAM."""
    def __init__(self, interval_seconds=0.05):
        self.interval_seconds = float(interval_seconds)
        self.process = psutil.Process(os.getpid())
        self.stop_event = threading.Event()
        self.thread = None
        self.baseline_rss_mb = 0.0
        self.peak_rss_mb = 0.0

    def _sample(self):
        try:
            rss_mb = self.process.memory_info().rss / (1024 ** 2)
            self.peak_rss_mb = max(self.peak_rss_mb, rss_mb)
        except (psutil.Error, OSError):
            pass

    def _monitor_loop(self):
        while not self.stop_event.wait(self.interval_seconds):
            self._sample()

    def __enter__(self):
        self.baseline_rss_mb = self.process.memory_info().rss / (1024 ** 2)
        self.peak_rss_mb = self.baseline_rss_mb
        self.thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join()
        self._sample()

    @property
    def peak_increase_mb(self):
        return max(0.0, self.peak_rss_mb - self.baseline_rss_mb)


def _safe_id(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def _make_training_sample(
    csv_path,
    feature_columns,
    target_column,
    class_order,
    max_rows_per_class=10_000,
    chunk_size=100_000,
    random_state=42,
):
    """Reservoir-sample up to max_rows_per_class rows for each class."""
    rng = np.random.default_rng(random_state)
    columns = list(feature_columns) + [target_column]
    reservoirs = {label: None for label in class_order}
    seen = {label: 0 for label in class_order}
    stored = {label: 0 for label in class_order}

    for chunk in pd.read_csv(csv_path, usecols=columns, chunksize=chunk_size):
        chunk[target_column] = chunk[target_column].astype(str)
        for label in class_order:
            class_rows = chunk.loc[chunk[target_column] == str(label), columns]
            if class_rows.empty:
                continue
            values = class_rows[feature_columns].to_numpy(dtype=np.float32, copy=True)
            labels = class_rows[target_column].to_numpy(dtype=object, copy=True)
            if reservoirs[label] is None:
                reservoirs[label] = np.empty((max_rows_per_class, len(feature_columns)), dtype=np.float32)
            seen_before = seen[label]
            stored_before = stored[label]
            fill_count = min(max_rows_per_class - stored_before, len(values))
            if fill_count > 0:
                reservoirs[label][stored_before:stored_before + fill_count] = values[:fill_count]
                stored[label] += fill_count

            for local_i in range(fill_count, len(values)):
                population_index = seen_before + local_i
                replacement_index = int(rng.integers(0, population_index + 1))
                if replacement_index < max_rows_per_class:
                    reservoirs[label][replacement_index] = values[local_i]
            seen[label] += len(values)

    missing = [str(label) for label in class_order if stored[label] == 0]
    if missing:
        raise ValueError(f"No training examples found for classes: {missing}")

    X_parts, y_parts = [], []
    for label in class_order:
        count = stored[label]
        X_parts.append(reservoirs[label][:count])
        y_parts.extend([str(label)] * count)

    X = np.vstack(X_parts).astype(np.float32, copy=False)
    y = np.asarray(y_parts, dtype=object)
    permutation = rng.permutation(len(y))
    X, y = X[permutation], y[permutation]

    summary = pd.DataFrame([
        {
            "label": str(label),
            "rows_seen_in_train_split": int(seen[label]),
            "rows_in_training_sample": int(stored[label]),
            "cap_per_class": int(max_rows_per_class),
        }
        for label in class_order
    ])
    return X, y, summary


def _make_estimator(model_id, seed, n_classes):
    if model_id == "decision_tree":
        return DecisionTreeClassifier(
            random_state=seed,
            max_depth=24,
            min_samples_leaf=5,
            class_weight="balanced",
        )
    if model_id == "random_forest":
        return RandomForestClassifier(
            n_estimators=100,
            random_state=seed,
            max_depth=20,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=-1,
        )
    if model_id == "xgboost":
        if not XGBOOST_AVAILABLE:
            raise ImportError("XGBoost is required. Install it in this environment with: pip install xgboost")
        return XGBClassifier(
            objective="multi:softprob",
            num_class=n_classes,
            n_estimators=100,
            max_depth=8,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=seed,
            n_jobs=min(8, os.cpu_count() or 1),
            eval_metric="mlogloss",
        )
    if model_id == "logistic_regression":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=500,
                C=1.0,
                class_weight="balanced",
                solver="lbfgs",
                random_state=seed,
            )),
        ])
    raise ValueError(f"Unknown model_id: {model_id}")


def _evaluate_predictions(y_true, y_pred, class_order):
    labels = np.arange(len(class_order))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    per_class = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=[str(c) for c in class_order],
        output_dict=True,
        zero_division=0,
    )
    per_class_rows = []
    for class_index, label in enumerate(class_order):
        row = per_class[str(label)]
        support = int(row["support"])
        per_class_rows.append({
            "label": str(label),
            "class_index": int(class_index),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1_score": float(row["f1-score"]),
            "support": support,
        })

    cm_rows = []
    row_sums = cm.sum(axis=1)
    for true_i, true_label in enumerate(class_order):
        for pred_i, pred_label in enumerate(class_order):
            count = int(cm[true_i, pred_i])
            denominator = int(row_sums[true_i])
            cm_rows.append({
                "true_label": str(true_label),
                "predicted_label": str(pred_label),
                "true_class_index": int(true_i),
                "predicted_class_index": int(pred_i),
                "count": count,
                "row_percentage": float(100.0 * count / denominator) if denominator else 0.0,
            })

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }
    return metrics, per_class_rows, cm_rows


def _atomic_csv_write(df, path):
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(temp_path, index=False)
    temp_path.replace(path)


def _atomic_json_write(data, path):
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    temp_path.replace(path)


print("Helpers defined.")

Helpers defined.


## 4. Build the fixed training sample and load the test split

The test set is not used for feature selection, preprocessing fitting, or training. The logistic-regression scaler is fitted inside its pipeline using the training sample only.

In [5]:
X_train_sample, y_train_labels, training_sample_summary = _make_training_sample(
    csv_path=TRAIN_PATH,
    feature_columns=MODEL_FEATURES,
    target_column=TARGET_COLUMN,
    class_order=CLASS_ORDER,
    max_rows_per_class=MAX_ROWS_PER_CLASS,
    chunk_size=CHUNK_SIZE,
    random_state=SAMPLE_RANDOM_STATE,
)

test_columns = list(MODEL_FEATURES) + [TARGET_COLUMN]
test_df = pd.read_csv(TEST_PATH, usecols=test_columns)
test_df[TARGET_COLUMN] = test_df[TARGET_COLUMN].astype(str)

unknown_test_labels = sorted(set(test_df[TARGET_COLUMN]) - set(map(str, CLASS_ORDER)))
unknown_train_labels = sorted(set(y_train_labels) - set(map(str, CLASS_ORDER)))
if unknown_test_labels or unknown_train_labels:
    raise ValueError(
        f"Labels differ from manifest class_order. "
        f"Unknown train labels: {unknown_train_labels}; unknown test labels: {unknown_test_labels}"
    )

label_to_index = {str(label): i for i, label in enumerate(CLASS_ORDER)}
y_train = np.array([label_to_index[str(label)] for label in y_train_labels], dtype=np.int32)
y_test = np.array([label_to_index[str(label)] for label in test_df[TARGET_COLUMN]], dtype=np.int32)

print("Training sample shape:", X_train_sample.shape)
print("Test split shape:", test_df[MODEL_FEATURES].shape)
print("Training sample rows by class:")
display(training_sample_summary)

training_sample_path = PROCESSED_DIR / "model_training_sample.csv.gz"
sample_to_save = pd.DataFrame(X_train_sample, columns=MODEL_FEATURES)
sample_to_save[TARGET_COLUMN] = y_train_labels
sample_to_save.to_csv(training_sample_path, index=False, compression="gzip")
training_sample_summary.to_csv(PHASE05_DIR / "training_sample_summary.csv", index=False)
print("Saved fixed training sample:", training_sample_path)

Training sample shape: (70245, 70)
Test split shape: (505044, 70)
Training sample rows by class:


,label,rows_seen_in_train_split,rows_in_training_sample,cap_per_class
0,Benign,1677531,10000,10000
1,Bot,1580,1580,10000
2,DDoS,102474,10000,10000
3,DoS GoldenEye,8191,8191,10000
4,DoS Hulk,138293,10000,10000
5,DoS Slowhttptest,4168,4168,10000
6,DoS slowloris,4317,4317,10000
7,FTP-Patator,4751,4751,10000
8,PortScan,72003,10000,10000
9,SSH-Patator,2614,2614,10000


Saved fixed training sample: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\data\processed\model_training_sample.csv.gz


## 5. Run the 200 model configurations

Each configuration gets its own model artifact and checkpointed results. With `RESUME=True`, a configuration is skipped only if its model artifact and its expected per-class and confusion-matrix rows are already present.

**Runtime note:** evaluating 200 configurations against the full test split is computationally expensive. Do not change the test split or the fixed seed list between configurations.

In [6]:
MODEL_SPECS = [
    ("decision_tree", "Decision Tree"),
    ("random_forest", "Random Forest"),
    ("xgboost", "XGBoost"),
    ("logistic_regression", "Logistic Regression"),
]

SEEDS = tuple(DEFAULT_SEEDS)
RESUME = True

if len(SEEDS) != 5 or len(set(SEEDS)) != 5:
    raise ValueError("Exactly five distinct seeds are required.")
if any(not isinstance(seed, (int, np.integer)) for seed in SEEDS):
    raise ValueError("All seeds must be integers.")
if not XGBOOST_AVAILABLE:
    raise ImportError(
        "XGBoost is not installed, but it is part of the planned 4-model experiment. "
        "Install it in the notebook environment with `%pip install xgboost`, then restart the kernel."
    )

SUMMARY_PATH = PHASE05_DIR / "experiment_summary.csv"
PER_CLASS_PATH = PHASE05_DIR / "per_class_metrics.csv"
CONFUSION_PATH = PHASE05_DIR / "confusion_matrix_long.csv"
MANIFEST_PATH = PHASE05_DIR / "model_training_manifest.json"

def _read_csv_if_exists(path):
    return pd.read_csv(path) if Path(path).exists() else pd.DataFrame()

# Keep only rows for this explicitly declared experiment matrix. This prevents
# checkpoints from an earlier seed list from contaminating the current exports.
expected_experiment_ids = {
    f"{_safe_id(fs['feature_set_id'])}__{_safe_id(model_id)}__seed_{seed}"
    for fs in FEATURE_SETS
    for model_id, _ in MODEL_SPECS
    for seed in SEEDS
}

summary_df = _read_csv_if_exists(SUMMARY_PATH)
per_class_df = _read_csv_if_exists(PER_CLASS_PATH)
confusion_df = _read_csv_if_exists(CONFUSION_PATH)

for frame in (summary_df, per_class_df, confusion_df):
    if not frame.empty and "experiment_id" in frame.columns:
        frame.drop(frame.index[~frame["experiment_id"].astype(str).isin(expected_experiment_ids)], inplace=True)

# Remove malformed/duplicate checkpoint rows, retaining the latest row for each key.
if not summary_df.empty and "experiment_id" in summary_df:
    summary_df = summary_df.drop_duplicates("experiment_id", keep="last")
if not per_class_df.empty and "experiment_id" in per_class_df:
    per_class_df = per_class_df.drop_duplicates(
        ["experiment_id", "label"], keep="last"
    )
if not confusion_df.empty and "experiment_id" in confusion_df:
    confusion_df = confusion_df.drop_duplicates(
        ["experiment_id", "true_label", "predicted_label"], keep="last"
    )

for feature_set in FEATURE_SETS:
    feature_names = feature_set["features"]
    feature_indices = [MODEL_FEATURES.index(name) for name in feature_names]
    X_train_set = X_train_sample[:, feature_indices]
    X_test_set = test_df[feature_names].to_numpy(dtype=np.float32, copy=True)

    for model_id, model_name in MODEL_SPECS:
        for seed in SEEDS:
            experiment_id = (
                f"{_safe_id(feature_set['feature_set_id'])}__"
                f"{_safe_id(model_id)}__seed_{seed}"
            )
            model_path = MODEL_DIR / f"{experiment_id}.joblib"

            summary_done = (
                not summary_df.empty
                and "experiment_id" in summary_df
                and (summary_df["experiment_id"].astype(str) == experiment_id).any()
            )
            per_class_done = (
                not per_class_df.empty
                and "experiment_id" in per_class_df
                and (per_class_df["experiment_id"].astype(str) == experiment_id).sum() == len(CLASS_ORDER)
            )
            confusion_done = (
                not confusion_df.empty
                and "experiment_id" in confusion_df
                and (confusion_df["experiment_id"].astype(str) == experiment_id).sum() == len(CLASS_ORDER) ** 2
            )

            if RESUME and model_path.exists() and summary_done and per_class_done and confusion_done:
                print("Skipping completed:", experiment_id)
                continue

            configuration_start = time.perf_counter()
            estimator = _make_estimator(model_id, int(seed), len(CLASS_ORDER))

            training_start = time.perf_counter()
            with PeakRSSMonitor(interval_seconds=0.05) as training_memory:
                if model_id == "xgboost":
                    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
                    estimator.fit(X_train_set, y_train, sample_weight=sample_weights)
                else:
                    estimator.fit(X_train_set, y_train)
            training_time_seconds = time.perf_counter() - training_start

            inference_start = time.perf_counter()
            with PeakRSSMonitor(interval_seconds=0.05) as inference_memory:
                predictions = np.asarray(estimator.predict(X_test_set), dtype=np.int32)
            inference_time_seconds = time.perf_counter() - inference_start

            evaluation_start = time.perf_counter()
            metrics, current_per_class, current_confusion = _evaluate_predictions(
                y_test, predictions, CLASS_ORDER
            )
            evaluation_time_seconds = time.perf_counter() - evaluation_start
            total_configuration_time_seconds = time.perf_counter() - configuration_start

            model_save_start = time.perf_counter()
            temp_model_path = model_path.with_suffix(".joblib.tmp")
            joblib.dump(estimator, temp_model_path, compress=3)
            temp_model_path.replace(model_path)
            model_save_time_seconds = time.perf_counter() - model_save_start

            result_row = {
                "experiment_id": experiment_id,
                "model_id": model_id,
                "model_name": model_name,
                "feature_set_id": feature_set["feature_set_id"],
                "feature_set_name": feature_set["feature_set_name"],
                "selection_method": feature_set["selection_method"],
                "feature_count": int(len(feature_names)),
                "seed": int(seed),
                "train_sample_rows": int(len(y_train)),
                "test_rows": int(len(y_test)),
                **metrics,
                "training_time_seconds": float(training_time_seconds),
                "inference_time_seconds": float(inference_time_seconds),
                "evaluation_time_seconds": float(evaluation_time_seconds),
                "total_configuration_time_seconds": float(total_configuration_time_seconds),
                "model_save_time_seconds": float(model_save_time_seconds),
                "peak_training_ram_mb": float(training_memory.peak_rss_mb),
                "training_ram_increase_mb": float(training_memory.peak_increase_mb),
                "peak_inference_ram_mb": float(inference_memory.peak_rss_mb),
                "inference_ram_increase_mb": float(inference_memory.peak_increase_mb),
                "inference_ms_per_flow": float(1000.0 * inference_time_seconds / len(y_test)),
                "model_artifact": str(model_path.relative_to(PROJECT_ROOT)),
            }

            for row in current_per_class:
                row.update({
                    "experiment_id": experiment_id,
                    "model_id": model_id,
                    "model_name": model_name,
                    "feature_set_id": feature_set["feature_set_id"],
                    "feature_set_name": feature_set["feature_set_name"],
                    "selection_method": feature_set["selection_method"],
                    "feature_count": int(len(feature_names)),
                    "seed": int(seed),
                })
            for row in current_confusion:
                row.update({
                    "experiment_id": experiment_id,
                    "model_id": model_id,
                    "model_name": model_name,
                    "feature_set_id": feature_set["feature_set_id"],
                    "feature_set_name": feature_set["feature_set_name"],
                    "selection_method": feature_set["selection_method"],
                    "feature_count": int(len(feature_names)),
                    "seed": int(seed),
                })

            summary_df = pd.concat(
                [summary_df[summary_df.get("experiment_id", pd.Series(dtype=str)).astype(str) != experiment_id],
                 pd.DataFrame([result_row])],
                ignore_index=True,
            )
            if not per_class_df.empty and "experiment_id" in per_class_df:
                per_class_df = per_class_df[per_class_df["experiment_id"].astype(str) != experiment_id]
            if not confusion_df.empty and "experiment_id" in confusion_df:
                confusion_df = confusion_df[confusion_df["experiment_id"].astype(str) != experiment_id]
            per_class_df = pd.concat([per_class_df, pd.DataFrame(current_per_class)], ignore_index=True)
            confusion_df = pd.concat([confusion_df, pd.DataFrame(current_confusion)], ignore_index=True)

            _atomic_csv_write(summary_df, SUMMARY_PATH)
            _atomic_csv_write(per_class_df, PER_CLASS_PATH)
            _atomic_csv_write(confusion_df, CONFUSION_PATH)

            print(
                f"Completed {len(summary_df):>3} summary rows | {experiment_id} | "
                f"macro-F1={metrics['macro_f1']:.4f} | "
                f"fit={training_time_seconds:.1f}s | "
                f"fit peak RSS={training_memory.peak_rss_mb:.1f} MB"
            )

manifest = {
    "phase": "05_model_training",
    "seed_master_random_state": SEED_MASTER_RANDOM_STATE,
    "seed_selection_note": (
        "Five distinct seeds were generated reproducibly from NumPy default_rng "
        "with the master seed, then fixed explicitly for this experiment."
    ),
    "seeds": [int(s) for s in SEEDS],
    "sample_random_state": SAMPLE_RANDOM_STATE,
    "max_rows_per_class": MAX_ROWS_PER_CLASS,
    "chunk_size": CHUNK_SIZE,
    "target_column": TARGET_COLUMN,
    "model_feature_count": len(MODEL_FEATURES),
    "class_order": [str(c) for c in CLASS_ORDER],
    "feature_sets": [
        {
            "feature_set_id": fs["feature_set_id"],
            "feature_set_name": fs["feature_set_name"],
            "selection_method": fs["selection_method"],
            "feature_count": fs["feature_count"],
        }
        for fs in FEATURE_SETS
    ],
    "models": [{"model_id": mid, "model_name": name} for mid, name in MODEL_SPECS],
    "expected_configuration_count": len(FEATURE_SETS) * len(MODEL_SPECS) * len(SEEDS),
    "completed_configuration_count": int(summary_df["experiment_id"].nunique()) if not summary_df.empty else 0,
    "training_sample_path": str(training_sample_path.relative_to(PROJECT_ROOT)),
    "summary_path": str(SUMMARY_PATH.relative_to(PROJECT_ROOT)),
    "per_class_metrics_path": str(PER_CLASS_PATH.relative_to(PROJECT_ROOT)),
    "confusion_matrix_path": str(CONFUSION_PATH.relative_to(PROJECT_ROOT)),
    "memory_measurement": (
        "Peak process RSS sampled every 0.05 seconds during fit and predict; "
        "RAM values include process and resident data, not model-only allocation."
    ),
}
_atomic_json_write(manifest, MANIFEST_PATH)

print("\nTraining loop finished or resumed.")
print("Completed configurations:", summary_df["experiment_id"].nunique() if not summary_df.empty else 0)
print("Expected configurations:", manifest["expected_configuration_count"])
print("Summary:", SUMMARY_PATH)
print("Per-class metrics:", PER_CLASS_PATH)
print("Confusion matrix:", CONFUSION_PATH)
print("Manifest:", MANIFEST_PATH)

Completed   1 summary rows | rf_top10__decision_tree__seed_425828 | macro-F1=0.8276 | fit=2.0s | fit peak RSS=521.3 MB
Completed   2 summary rows | rf_top10__decision_tree__seed_142707 | macro-F1=0.8261 | fit=2.0s | fit peak RSS=523.8 MB
Completed   3 summary rows | rf_top10__decision_tree__seed_692819 | macro-F1=0.8267 | fit=2.1s | fit peak RSS=523.8 MB
Completed   4 summary rows | rf_top10__decision_tree__seed_2184 | macro-F1=0.8276 | fit=2.0s | fit peak RSS=523.8 MB
Completed   5 summary rows | rf_top10__decision_tree__seed_136548 | macro-F1=0.8275 | fit=2.0s | fit peak RSS=523.8 MB
Completed   6 summary rows | rf_top10__random_forest__seed_425828 | macro-F1=0.8747 | fit=18.5s | fit peak RSS=561.6 MB
Completed   7 summary rows | rf_top10__random_forest__seed_142707 | macro-F1=0.8783 | fit=20.9s | fit peak RSS=564.4 MB
Completed   8 summary rows | rf_top10__random_forest__seed_692819 | macro-F1=0.8764 | fit=23.0s | fit peak RSS=563.3 MB
Completed   9 summary rows | rf_top10__random_f

C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Completed  16 summary rows | rf_top10__logistic_regression__seed_425828 | macro-F1=0.3804 | fit=30.3s | fit peak RSS=550.6 MB


C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Completed  17 summary rows | rf_top10__logistic_regression__seed_142707 | macro-F1=0.3804 | fit=27.0s | fit peak RSS=585.8 MB


...

Completed 198 summary rows | all_features__logistic_regression__seed_692819 | macro-F1=0.5692 | fit=101.0s | fit peak RSS=833.8 MB


C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Completed 199 summary rows | all_features__logistic_regression__seed_2184 | macro-F1=0.5692 | fit=101.1s | fit peak RSS=833.9 MB


C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Completed 200 summary rows | all_features__logistic_regression__seed_136548 | macro-F1=0.5692 | fit=105.3s | fit peak RSS=835.4 MB

Training loop finished or resumed.
Completed configurations: 200
Expected configurations: 200
Summary: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_model_results\experiment_summary.csv
Per-class metrics: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_model_results\per_class_metrics.csv
Confusion matrix: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_model_results\confusion_matrix_long.csv
Manifest: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_model_results\model_training_manifest.json


## 6. Check experiment completeness and inspect results

Expected result sizes for a complete run: 200 experiment-summary rows, 2,400 per-class rows (200 × 12 classes), and 28,800 confusion-matrix rows (200 × 12 × 12).

In [3]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    r"C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_model_results"
)

SUMMARY_PATH = OUTPUT_DIR / "experiment_summary.csv"
PER_CLASS_PATH = OUTPUT_DIR / "per_class_metrics.csv"
CONFUSION_PATH = OUTPUT_DIR / "confusion_matrix_long.csv"
MANIFEST_PATH = OUTPUT_DIR / "model_training_manifest.json"

for path in [SUMMARY_PATH, PER_CLASS_PATH, CONFUSION_PATH, MANIFEST_PATH]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    print("\nSummary rows:", len(summary_df))

    if "experiment_id" in summary_df.columns:
        print("Unique configurations:", summary_df["experiment_id"].nunique())

    display(summary_df.head())

experiment_summary.csv: FOUND
per_class_metrics.csv: FOUND
confusion_matrix_long.csv: FOUND
model_training_manifest.json: FOUND

Summary rows: 200
Unique configurations: 200


,experiment_id,model_id,model_name,feature_set_id,feature_set_name,selection_method,feature_count,seed,training_sample_rows,test_rows,...,model_artifact_path,train_sample_rows,evaluation_time_seconds,total_configuration_time_seconds,model_save_time_seconds,peak_training_ram_mb,training_ram_increase_mb,peak_inference_ram_mb,inference_ram_increase_mb,model_artifact
0,rf_top10__decision_tree__seed_425828,decision_tree,Decision Tree,rf_top10,Top 10 — Random Forest Importance,rf_importance,10,425828,NaN,505044,...,NaN,70245.0,1.728385,4.025724,0.020851,521.261719,2.367188,571.855469,50.593750,results\models\phase05\rf_top10__decision_tree...
1,rf_top10__decision_tree__seed_142707,decision_tree,Decision Tree,rf_top10,Top 10 — Random Forest Importance,rf_importance,10,142707,NaN,505044,...,NaN,70245.0,1.612213,3.948507,0.027050,523.816406,0.042969,573.589844,49.792969,results\models\phase05\rf_top10__decision_tree...
2,rf_top10__decision_tree__seed_692819,decision_tree,Decision Tree,rf_top10,Top 10 — Random Forest Importance,rf_importance,10,692819,NaN,505044,...,NaN,70245.0,1.750235,4.289964,0.029597,523.812500,0.019531,574.777344,50.968750,results\models\phase05\rf_top10__decision_tree...
3,rf_top10__decision_tree__seed_2184,decision_tree,Decision Tree,rf_top10,Top 10 — Random Forest Importance,rf_importance,10,2184,NaN,505044,...,NaN,70245.0,2.014319,4.216163,0.018966,523.820312,0.019531,570.519531,46.718750,results\models\phase05\rf_top10__decision_tree...
4,rf_top10__decision_tree__seed_136548,decision_tree,Decision Tree,rf_top10,Top 10 — Random Forest Importance,rf_importance,10,136548,NaN,505044,...,NaN,70245.0,1.615959,3.935070,0.027257,523.820312,0.019531,571.226562,47.410156,results\models\phase05\rf_top10__decision_tree...


In [5]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    r"C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_model_results"
)

summary_df = pd.read_csv(OUTPUT_DIR / "experiment_summary.csv")
per_class_df = pd.read_csv(OUTPUT_DIR / "per_class_metrics.csv")
confusion_df = pd.read_csv(OUTPUT_DIR / "confusion_matrix_long.csv")

# 10 feature sets × 4 models × 5 seeds
expected_experiments = 10 * 4 * 5

# Infer the number of class rows and confusion-matrix cells
# per experiment from the saved results.
per_class_counts = per_class_df.groupby("experiment_id").size()
confusion_counts = confusion_df.groupby("experiment_id").size()

classes_per_experiment = int(per_class_counts.mode().iloc[0])
confusion_cells_per_experiment = int(confusion_counts.mode().iloc[0])

completeness = {
    "summary_rows": len(summary_df),
    "unique_experiments": summary_df["experiment_id"].nunique(),
    "expected_experiments": expected_experiments,
    "per_class_rows": len(per_class_df),
    "expected_per_class_rows": expected_experiments * classes_per_experiment,
    "confusion_matrix_rows": len(confusion_df),
    "expected_confusion_matrix_rows": (
        expected_experiments * classes_per_experiment**2
    ),
    "experiments_with_unusual_per_class_row_count": int(
        (per_class_counts != classes_per_experiment).sum()
    ),
    "experiments_with_unusual_confusion_row_count": int(
        (confusion_counts != confusion_cells_per_experiment).sum()
    ),
}

display(pd.DataFrame([completeness]))

display(
    summary_df.sort_values("macro_f1", ascending=False)[
        [
            "model_name", "feature_set_name", "feature_count", "seed",
            "accuracy", "macro_f1", "weighted_f1",
            "training_time_seconds", "inference_time_seconds",
            "peak_training_ram_mb", "training_ram_increase_mb",
        ]
    ].head(20)
)

,summary_rows,unique_experiments,expected_experiments,per_class_rows,expected_per_class_rows,confusion_matrix_rows,expected_confusion_matrix_rows,experiments_with_unusual_per_class_row_count,experiments_with_unusual_confusion_row_count
0,200,200,200,2400,2400,28800,28800,0,0


,model_name,feature_set_name,feature_count,seed,accuracy,macro_f1,weighted_f1,training_time_seconds,inference_time_seconds,peak_training_ram_mb,training_ram_increase_mb
193,XGBoost,All 70 Features (Shared Baseline),70,2184,0.997268,0.944746,0.997485,124.361616,15.458016,742.328125,51.636719
173,XGBoost,Top 40 — RFE (Decision Tree),40,2184,0.997313,0.944325,0.997529,98.842829,15.098955,592.562500,45.199219
194,XGBoost,All 70 Features (Shared Baseline),70,136548,0.997186,0.943708,0.997403,119.168814,14.940321,740.597656,49.625000
170,XGBoost,Top 40 — RFE (Decision Tree),40,425828,0.997313,0.943045,0.997535,97.319055,15.429926,585.730469,39.828125
190,XGBoost,All 70 Features (Shared Baseline),70,425828,0.997188,0.942102,0.997409,116.694011,15.653638,779.480469,60.347656
172,XGBoost,Top 40 — RFE (Decision Tree),40,692819,0.997198,0.941974,0.997440,101.655058,15.683910,579.542969,32.210938
171,XGBoost,Top 40 — RFE (Decision Tree),40,142707,0.997204,0.941175,0.997432,101.039151,15.611982,583.917969,36.605469
191,XGBoost,All 70 Features (Shared Baseline),70,142707,0.997299,0.940867,0.997534,121.131564,14.939310,732.457031,65.652344
112,XGBoost,Top 40 — ANOVA F-test,40,692819,0.997139,0.940838,0.997350,97.687700,14.976496,557.664062,32.050781
192,XGBoost,All 70 Features (Shared Baseline),70,692819,0.997070,0.940680,0.997312,119.831344,15.153968,743.781250,54.011719


## 7. Export seed-aggregated visualization tables

These tables are intended for Tableau/Power BI. Means and standard deviations are calculated across the five fixed seeds for each model × feature-set combination. Confusion-matrix counts and row percentages are averaged across seeds.

In [7]:
from pathlib import Path

OUTPUT_DIR = Path(
    r"C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_model_results"
)

VISUALIZATION_DIR = OUTPUT_DIR.parent / "phase05_visualization"
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory exists:", OUTPUT_DIR.exists())
print("Visualization directory:", VISUALIZATION_DIR)

Output directory exists: True
Visualization directory: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_visualization


In [8]:
def _mean_std_agg(column):
    return [
        (f"mean_{column}", (column, "mean")),
        (f"std_{column}", (column, "std")),
    ]

group_keys = [
    "model_id", "model_name", "feature_set_id",
    "feature_set_name", "selection_method", "feature_count",
]

agg_spec = {
    "seed_count": ("seed", "nunique"),
    "mean_accuracy": ("accuracy", "mean"),
    "std_accuracy": ("accuracy", "std"),
    "mean_macro_precision": ("macro_precision", "mean"),
    "std_macro_precision": ("macro_precision", "std"),
    "mean_macro_recall": ("macro_recall", "mean"),
    "std_macro_recall": ("macro_recall", "std"),
    "mean_macro_f1": ("macro_f1", "mean"),
    "std_macro_f1": ("macro_f1", "std"),
    "mean_weighted_f1": ("weighted_f1", "mean"),
    "std_weighted_f1": ("weighted_f1", "std"),
    "mean_training_time_seconds": ("training_time_seconds", "mean"),
    "std_training_time_seconds": ("training_time_seconds", "std"),
    "mean_inference_time_seconds": ("inference_time_seconds", "mean"),
    "std_inference_time_seconds": ("inference_time_seconds", "std"),
    "mean_evaluation_time_seconds": ("evaluation_time_seconds", "mean"),
    "mean_total_configuration_time_seconds": ("total_configuration_time_seconds", "mean"),
    "std_total_configuration_time_seconds": ("total_configuration_time_seconds", "std"),
    "mean_model_save_time_seconds": ("model_save_time_seconds", "mean"),
    "mean_peak_training_ram_mb": ("peak_training_ram_mb", "mean"),
    "std_peak_training_ram_mb": ("peak_training_ram_mb", "std"),
    "mean_training_ram_increase_mb": ("training_ram_increase_mb", "mean"),
    "mean_peak_inference_ram_mb": ("peak_inference_ram_mb", "mean"),
    "mean_inference_ram_increase_mb": ("inference_ram_increase_mb", "mean"),
    "mean_inference_ms_per_flow": ("inference_ms_per_flow", "mean"),
}
metrics_by_model_feature_set = (
    summary_df.groupby(group_keys, dropna=False)
    .agg(**agg_spec)
    .reset_index()
)
metrics_path = VISUALIZATION_DIR / "metrics_by_model_and_feature_set.csv"
metrics_by_model_feature_set.to_csv(metrics_path, index=False)

per_class_group_keys = [
    "model_id", "model_name", "feature_set_id", "feature_set_name",
    "selection_method", "feature_count", "label", "class_index",
]
per_class_agg = (
    per_class_df.groupby(per_class_group_keys, dropna=False)
    .agg(
        seed_count=("seed", "nunique"),
        mean_precision=("precision", "mean"),
        std_precision=("precision", "std"),
        mean_recall=("recall", "mean"),
        std_recall=("recall", "std"),
        mean_f1_score=("f1_score", "mean"),
        std_f1_score=("f1_score", "std"),
        mean_support=("support", "mean"),
    )
    .reset_index()
)
per_class_agg.to_csv(VISUALIZATION_DIR / "per_class_metrics_across_seeds.csv", index=False)

confusion_group_keys = [
    "model_id", "model_name", "feature_set_id", "feature_set_name",
    "selection_method", "feature_count", "true_label", "predicted_label",
    "true_class_index", "predicted_class_index",
]
confusion_agg = (
    confusion_df.groupby(confusion_group_keys, dropna=False)
    .agg(
        seed_count=("seed", "nunique"),
        mean_count=("count", "mean"),
        std_count=("count", "std"),
        mean_row_percentage=("row_percentage", "mean"),
        std_row_percentage=("row_percentage", "std"),
    )
    .reset_index()
)
confusion_agg.to_csv(VISUALIZATION_DIR / "confusion_matrix_across_seeds.csv", index=False)

print("Saved:", metrics_path, "| rows:", len(metrics_by_model_feature_set))
print("Saved:", VISUALIZATION_DIR / "per_class_metrics_across_seeds.csv", "| rows:", len(per_class_agg))
print("Saved:", VISUALIZATION_DIR / "confusion_matrix_across_seeds.csv", "| rows:", len(confusion_agg))
display(metrics_by_model_feature_set.sort_values("mean_macro_f1", ascending=False).head(15))

Saved: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_visualization\metrics_by_model_and_feature_set.csv | rows: 40
Saved: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_visualization\per_class_metrics_across_seeds.csv | rows: 480
Saved: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase05_visualization\confusion_matrix_across_seeds.csv | rows: 5760


,model_id,model_name,feature_set_id,feature_set_name,selection_method,feature_count,seed_count,mean_accuracy,std_accuracy,mean_macro_precision,...,mean_evaluation_time_seconds,mean_total_configuration_time_seconds,std_total_configuration_time_seconds,mean_model_save_time_seconds,mean_peak_training_ram_mb,std_peak_training_ram_mb,mean_training_ram_increase_mb,mean_peak_inference_ram_mb,mean_inference_ram_increase_mb,mean_inference_ms_per_flow
30,xgboost,XGBoost,all_features,All 70 Features (Shared Baseline),shared_all_features,70,5,0.997202,0.000089,0.909413,...,1.398462,136.875153,2.781226,0.261097,747.728906,18.282499,56.254688,739.714844,43.783594,0.030154
39,xgboost,XGBoost,rfe_top40,Top 40 — RFE (Decision Tree),rfe_decision_tree,40,5,0.997224,0.000093,0.909473,...,1.407735,117.527855,2.727649,0.259613,585.510156,4.690647,38.450781,594.159375,46.874219,0.030621
33,xgboost,XGBoost,anova_top40,Top 40 — ANOVA F-test,anova_f_test,40,5,0.997006,0.000078,0.903440,...,1.382685,111.625712,5.229411,0.258093,635.728906,105.388936,33.871094,618.435156,39.292187,0.028903
38,xgboost,XGBoost,rfe_top20,Top 20 — RFE (Decision Tree),rfe_decision_tree,20,5,0.997089,0.000123,0.901431,...,1.360233,98.836855,9.085713,0.250466,472.648438,2.015282,24.875781,491.294531,43.385156,0.029301
28,random_forest,Random Forest,rfe_top20,Top 20 — RFE (Decision Tree),rfe_decision_tree,20,5,0.992805,0.000235,0.870757,...,1.680827,32.551054,0.965289,0.853417,451.089844,1.856079,21.193750,1040.825000,594.794531,0.022724
36,xgboost,XGBoost,rf_top40,Top 40 — Random Forest Importance,rf_importance,40,5,0.995252,0.000039,0.864353,...,1.484259,111.846414,4.757746,0.268502,756.170313,2.591628,36.740625,763.514063,43.865625,0.030818
35,xgboost,XGBoost,rf_top20,Top 20 — Random Forest Importance,rf_importance,20,5,0.994868,0.000048,0.862516,...,1.543738,94.059552,3.526914,0.280225,659.463281,1.326129,25.496875,680.025000,45.921875,0.029740
34,xgboost,XGBoost,rf_top10,Top 10 — Random Forest Importance,rf_importance,10,5,0.993789,0.000069,0.859171,...,1.389651,83.515410,7.733881,0.341769,547.957031,1.636362,31.008594,562.326563,31.010938,0.029309
29,random_forest,Random Forest,rfe_top40,Top 40 — RFE (Decision Tree),rfe_decision_tree,40,5,0.991601,0.000202,0.861155,...,1.848433,47.213211,1.349152,1.097438,552.922656,6.733798,20.769531,1129.457813,588.739844,0.025925
20,random_forest,Random Forest,all_features,All 70 Features (Shared Baseline),shared_all_features,70,5,0.990269,0.000488,0.859705,...,1.624512,46.553532,1.572862,0.846843,725.069531,5.610856,5.896094,1307.795312,587.861719,0.024961


## 8. Best configuration and class-level results

The best configuration below is selected by macro-F1 on the held-out test split. Because this is test-set-based comparison across many configurations, treat the ranking as an experiment result rather than an unbiased final estimate after model selection.

In [9]:
# Find the best individual configuration by Macro-F1
best_row = summary_df.sort_values(
    ["macro_f1", "weighted_f1", "accuracy"],
    ascending=False,
).iloc[0]

print("Best single configuration by macro-F1:")
display(best_row.to_frame("value"))

best_id = best_row["experiment_id"]

# Show per-class performance
best_per_class = (
    per_class_df.loc[per_class_df["experiment_id"] == best_id]
    .sort_values("class_index")
)

display(
    best_per_class[
        ["label", "precision", "recall", "f1_score", "support"]
    ]
)

# Build the confusion matrix using class indices,
# avoiding dependence on the undefined CLASS_ORDER variable.
best_classes = (
    best_per_class[["label", "class_index"]]
    .drop_duplicates()
    .sort_values("class_index")
)

class_indices = best_classes["class_index"].astype(int).tolist()
class_labels = best_classes["label"].astype(str).tolist()

best_cm = (
    confusion_df.loc[confusion_df["experiment_id"] == best_id]
    .pivot(
        index="true_class_index",
        columns="predicted_class_index",
        values="count",
    )
    .reindex(index=class_indices, columns=class_indices)
)

best_cm.index = class_labels
best_cm.columns = class_labels

print("Confusion matrix for the best configuration:")
display(best_cm)

Best single configuration by macro-F1:


,value
experiment_id,all_features__xgboost__seed_2184
model_id,xgboost
model_name,XGBoost
feature_set_id,all_features
feature_set_name,All 70 Features (Shared Baseline)
selection_method,shared_all_features
feature_count,70
seed,2184
training_sample_rows,NaN
test_rows,505044


,label,precision,recall,f1_score,support
2316,Benign,0.999962,0.996946,0.998452,419508.0
2317,Bot,0.423299,0.997283,0.594332,368.0
2318,DDoS,0.998318,0.999334,0.998826,25537.0
2319,DoS GoldenEye,0.962282,0.998568,0.980089,2095.0
2320,DoS Hulk,0.991722,0.998838,0.995267,34424.0
2321,DoS Slowhttptest,0.910745,0.991509,0.949413,1060.0
2322,DoS slowloris,0.976103,0.995314,0.985615,1067.0
2323,FTP-Patator,0.932070,1.000000,0.964841,1180.0
2324,PortScan,0.994016,0.998897,0.996450,18127.0
2325,SSH-Patator,0.916667,1.000000,0.956522,605.0


Confusion matrix for the best configuration:


,Benign,Bot,DDoS,DoS GoldenEye,DoS Hulk,DoS Slowhttptest,DoS slowloris,FTP-Patator,PortScan,SSH-Patator,SYN Flood,Web Attack
Benign,418227,500,40,53,267,92,16,85,108,55,0,65
Bot,1,367,0,0,0,0,0,0,0,0,0,0
DDoS,11,0,25520,0,6,0,0,0,0,0,0,0
DoS GoldenEye,0,0,0,2092,0,3,0,0,0,0,0,0
DoS Hulk,2,0,3,26,34384,5,3,0,1,0,0,0
DoS Slowhttptest,1,0,0,1,0,1051,6,0,0,0,0,1
DoS slowloris,0,0,0,0,0,3,1062,0,0,0,0,2
FTP-Patator,0,0,0,0,0,0,0,1180,0,0,0,0
PortScan,1,0,0,0,14,0,0,0,18107,0,0,5
SSH-Patator,0,0,0,0,0,0,0,0,0,605,0,0


## 9. Output inventory

Phase 05 writes the following files:

- `exports/phase05_model_results/experiment_summary.csv`
- `exports/phase05_model_results/per_class_metrics.csv`
- `exports/phase05_model_results/confusion_matrix_long.csv`
- `exports/phase05_model_results/training_sample_summary.csv`
- `exports/phase05_model_results/model_training_manifest.json`
- `data/processed/model_training_sample.csv.gz`
- `results/models/phase05/*.joblib`
- `exports/phase05_visualization/metrics_by_model_and_feature_set.csv`
- `exports/phase05_visualization/per_class_metrics_across_seeds.csv`
- `exports/phase05_visualization/confusion_matrix_across_seeds.csv`

The model artifacts are retained for the later SHAP phase. `model_save_time_seconds` is recorded separately and is not included in `total_configuration_time_seconds`; that total covers estimator construction, fit, prediction, and metric evaluation.